In [1]:
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/datasets/ngocccchannn/warm-scenario/test_warm.parquet
/kaggle/input/datasets/ngocccchannn/warm-scenario/train_warm.parquet
/kaggle/input/datasets/ngocccchannn/warm-scenario/val_warm.parquet
/kaggle/input/datasets/ngocccchannn/evaluator/evaluate.py


In [2]:
import os
import sys 
import subprocess
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import pandas as pd
import json
from torch.utils.data import Dataset, DataLoader

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Sử dụng thiết bị: {DEVICE}")


sys.path.append('/kaggle/input/datasets/ngocccchannn/evaluator') 

from evaluate import evaluate

Sử dụng thiết bị: cuda


In [3]:
# Load dữ liệu warm
tr_df = pd.read_parquet('/kaggle/input/datasets/ngocccchannn/warm-scenario/train_warm.parquet')
va_df = pd.read_parquet('/kaggle/input/datasets/ngocccchannn/warm-scenario/val_warm.parquet')
te_df = pd.read_parquet('/kaggle/input/datasets/ngocccchannn/warm-scenario/test_warm.parquet')

# Xây dựng ánh xạ user2id và item2id CHỈ dựa trên tập Train
all_warm_users = tr_df['user_id'].unique()
all_warm_items = tr_df['course_id'].unique()

warm_user2id = {str(u): i for i, u in enumerate(all_warm_users)}
warm_item2id = {str(item): i for i, item in enumerate(all_warm_items)}

num_users = len(warm_user2id)
num_items = len(warm_item2id)
print(f"Tổng số Users (Train): {num_users}, Tổng số Items (Train): {num_items}")

# Chuẩn bị tensor tương tác cho tập train warm
train_users = torch.tensor([warm_user2id[str(u)] for u in tr_df['user_id'] if str(u) in warm_user2id], dtype=torch.long)
train_items = torch.tensor([warm_item2id[str(i)] for i in tr_df['course_id'] if str(i) in warm_item2id], dtype=torch.long)

# Lịch sử tương tác (chỉ lọc các user/item có trong tập Train)
hist_warm_df = pd.concat([tr_df, va_df])
hist_warm = {}
for u, group in hist_warm_df.groupby('user_id'):
    if str(u) in warm_user2id:
        u_idx = warm_user2id[str(u)]
        item_idxs = [warm_item2id[str(i)] for i in group['course_id'] if str(i) in warm_item2id]
        if len(item_idxs) > 0:
            hist_warm[u_idx] = item_idxs

target_warm = {}
for u, group in te_df.groupby('user_id'):
    if str(u) in warm_user2id:
        u_idx = warm_user2id[str(u)]
        item_idxs = [warm_item2id[str(i)] for i in group['course_id'] if str(i) in warm_item2id]
        if len(item_idxs) > 0:
            target_warm[u_idx] = item_idxs

Tổng số Users (Train): 88729, Tổng số Items (Train): 2448


In [4]:
# Hàm tính mức nền ngẫu nhiên kỳ vọng 
def expected_random_recall_support(target, mask, support, k=20):
    """support: mảng bool [n_items], các khóa mô hình có thể cho điểm khác hằng
    (ví dụ: khóa xuất hiện trong train). Khóa ngoài support coi như luôn ở cuối."""
    vals = []
    n_sup = int(support.sum())
    for u, gt in target.items():
        n = n_sup - sum(1 for i in mask.get(u, ()) if support[i])
        frac_in = sum(1 for i in gt if support[i]) / len(gt)
        vals.append(min(1.0, k / max(1, n)) * frac_in)
    return float(np.mean(vals))

In [5]:
# Tạo mảng support cho các item có trong train
train_item_indices = train_items.numpy()
support_train = np.zeros(num_items, dtype=bool)
support_train[train_item_indices] = True

ref = expected_random_recall_support(target_warm, hist_warm, support_train, k = 20)   # ≈ 0,008
fixed_threshold = 2 * ref
print(f"[Ngưỡng đặt trước] ref = {ref:.4f} | Ngưỡng an toàn (2 * ref) = {fixed_threshold:.4f}")

[Ngưỡng đặt trước] ref = 0.0083 | Ngưỡng an toàn (2 * ref) = 0.0165


In [6]:
class LightGCN(nn.Module):
    def __init__(self, num_users, num_items, latent_dim=64, n_layers=3):
        super(LightGCN, self).__init__()
        self.num_users = num_users
        self.num_items = num_items
        self.latent_dim = latent_dim
        self.n_layers = n_layers
        
        self.user_embedding = nn.Embedding(num_users, latent_dim)
        self.item_embedding = nn.Embedding(num_items, latent_dim)
        nn.init.xavier_uniform_(self.user_embedding.weight)
        nn.init.xavier_uniform_(self.item_embedding.weight)

    def forward(self, adj_matrix):
        users_emb = self.user_embedding.weight
        items_emb = self.item_embedding.weight
        all_emb = torch.cat([users_emb, items_emb], dim=0)
        
        embs = [all_emb]
        for _ in range(self.n_layers):
            all_emb = torch.sparse.mm(adj_matrix, all_emb)
            embs.append(all_emb)
            
        lightgcn_emb = torch.stack(embs, dim=1).mean(dim=1)
        users_emb_final, items_emb_final = torch.split(lightgcn_emb, [self.num_users, self.num_items])
        return users_emb_final, items_emb_final

In [7]:
def bpr_loss(users, pos_items, neg_items, u_emb, i_emb):
    u_g = u_emb[users]
    pos_g = i_emb[pos_items]
    neg_g = i_emb[neg_items]
    pos_scores = torch.sum(u_g * pos_g, dim=1)
    neg_scores = torch.sum(u_g * neg_g, dim=1)
    loss = -torch.mean(torch.nn.functional.logsigmoid(pos_scores - neg_scores))
    return loss

In [8]:
epochs = 40 
SEEDS = [42, 2023, 2024, 1007, 999]
lightgcn_shuffled_scores = {}

print("\n--- Bắt đầu huấn luyện LightGCN với nhãn xáo trộn (Shuffled Labels) ---")
for s in SEEDS:
    print(f"Đang huấn luyện với Seed: {s}...")
    torch.manual_seed(s)
    np.random.seed(s)
    
    perm_train = torch.randperm(len(train_users), generator=torch.Generator().manual_seed(s))
    shuffled_train_users = train_users
    shuffled_train_items = train_items[perm_train]
    
    row = torch.cat([shuffled_train_users, shuffled_train_items + num_users])
    col = torch.cat([shuffled_train_items + num_users, shuffled_train_users])
    val = torch.ones(row.size(0), dtype=torch.float32)
    
    assoc = torch.sparse_coo_tensor(torch.stack([row, col]), val, (num_users + num_items, num_users + num_items)).to(DEVICE)
    deg = torch.sparse.sum(assoc, dim=1).to_dense()
    deg_inv_sqrt = torch.pow(deg, -0.5)
    deg_inv_sqrt[deg_inv_sqrt == float('inf')] = 0.0
    
    i_idx = torch.arange(num_users + num_items, device=DEVICE)
    diag_indices = torch.stack([i_idx, i_idx], dim=0)
    deg_mat_inv_sqrt = torch.sparse_coo_tensor(diag_indices, deg_inv_sqrt.to(DEVICE), (num_users + num_items, num_users + num_items))
    adj_mat_s = torch.sparse.mm(deg_mat_inv_sqrt, torch.sparse.mm(assoc, deg_mat_inv_sqrt))
    
    model_shuffled = LightGCN(num_users, num_items, latent_dim=64, n_layers=3).to(DEVICE)
    optimizer_s = optim.Adam(model_shuffled.parameters(), lr=1e-3)
    
    model_shuffled.train()
    for epoch in range(epochs):
        optimizer_s.zero_grad()
        u_emb_s, i_emb_s = model_shuffled(adj_mat_s)
        
        perm_b = torch.randperm(len(shuffled_train_users))
        b_u = shuffled_train_users[perm_b][:2048].to(DEVICE)
        b_pos = shuffled_train_items[perm_b][:2048].to(DEVICE)
        b_neg = torch.randint(0, num_items, (2048,), dtype=torch.long).to(DEVICE)
        
        loss_s = bpr_loss(b_u, b_pos, b_neg, u_emb_s, i_emb_s)
        loss_s.backward()
        optimizer_s.step()
        
    model_shuffled.eval()
    with torch.no_grad():
        u_e, i_e = model_shuffled(adj_mat_s)
        def score_fn_s(users_tensor):
            u_idx = users_tensor.to(DEVICE)
            return torch.matmul(u_e[u_idx], i_e.t())
        lightgcn_shuffled_scores[s] = score_fn_s


--- Bắt đầu huấn luyện LightGCN với nhãn xáo trộn (Shuffled Labels) ---
Đang huấn luyện với Seed: 42...


/tmp/ipykernel_23/427076167.py:19: UserWarning: Sparse invariant checks are implicitly disabled. Memory errors (e.g. SEGFAULT) will occur when operating on a sparse tensor which violates the invariants, but checks incur performance overhead. To silence this warning, explicitly opt in or out. See `torch.sparse.check_sparse_tensor_invariants.__doc__` for guidance.  (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:760.)
  assoc = torch.sparse_coo_tensor(torch.stack([row, col]), val, (num_users + num_items, num_users + num_items)).to(DEVICE)


Đang huấn luyện với Seed: 2023...
Đang huấn luyện với Seed: 2024...
Đang huấn luyện với Seed: 1007...
Đang huấn luyện với Seed: 999...


In [9]:
# Ngưỡng đặt trước, ghi vào tài liệu trước khi chạy:
r20 = np.mean([evaluate(lightgcn_shuffled_scores[s], target_warm, hist_warm,
                        ks=(20,), tie_seed=s)["Recall@20"].mean() for s in SEEDS])

ok = r20 <= fixed_threshold
print(f"\n[Kết quả Rò rỉ] LightGCN nhãn xáo R@20 trung bình = {r20:.4f} | ngưỡng = {fixed_threshold:.4f}")
print(f"So với ngưỡng đặt trước (2 * ref) = {fixed_threshold:.4f} --> {'ĐẠT' if ok else 'KHÔNG ĐẠT'}")

assert ok, "Kiểm tra rò rỉ KHÔNG ĐẠT! Mô hình đang bị rò rỉ thông tin từ nhãn."


[Kết quả Rò rỉ] LightGCN nhãn xáo R@20 trung bình = 0.0752 | ngưỡng = 0.0165
So với ngưỡng đặt trước (2 * ref) = 0.0165 --> KHÔNG ĐẠT


AssertionError: Kiểm tra rò rỉ KHÔNG ĐẠT! Mô hình đang bị rò rỉ thông tin từ nhãn.